In [1]:
!pip install transformers datasets torch

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [3]:
# 1. Load the Yelp Review Full dataset (5-class sentiment, ~650k reviews)
# Only the review 'text' field is used — we ignore the star-rating label
dataset = load_dataset("yelp_review_full", split="train")
print(f"Training examples : {len(dataset)}")
print(f"Column names      : {dataset.column_names}")
print(f"\nSample review:\n{dataset[2]['text']}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training examples : 650000
Column names      : ['label', 'text']

Sample review:
Been going to Dr. Goldberg for over 10 years. I think I was one of his 1st patients when he started at MHMG. He's been great over the years and is really all about the big picture. It is because of him, not my now former gyn Dr. Markoff, that I found out I have fibroids. He explores all options with you and is very patient and understanding. He doesn't judge and asks all the right questions. Very thorough and wants to be kept in the loop on every aspect of your medical health and your life.


In [4]:
# 2. Use RoBERTa's tokenizer — byte-level BPE, case-sensitive, no [CLS]/[SEP] by default
# This is different from both GPT-2 (causal) and DistilBERT (WordPiece, lowercased)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"Vocab size        : {tokenizer.vocab_size}")
print(f"Model max length  : {tokenizer.model_max_length}")
print(f"Special tokens    : {tokenizer.all_special_tokens}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size        : 50265
Model max length  : 512
Special tokens    : ['<s>', '</s>', '<unk>', '<pad>', '<mask>']


In [5]:
# 3. Tokenize the dataset in batches using .map()
# add_special_tokens=False so we don't insert <s>/</s> between every article
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=False,
        add_special_tokens=False
    )

tokenized_ds = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text", "label"]
)

# Quick sanity check: decode a sample back to readable text
sample_ids = tokenized_ds[5]["input_ids"]
print(f"Columns           : {tokenized_ds.column_names}")
print(f"Example token count: {len(sample_ids)}")
print(f"First 20 token IDs: {sample_ids[:20]}")
print(f"Decoded           : {tokenizer.decode(sample_ids[:20])}")

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (987 > 512). Running this sequence through the model will result in indexing errors


Columns           : ['input_ids', 'attention_mask']
Example token count: 70
First 20 token IDs: [14323, 16046, 3299, 11, 10, 299, 16046, 1524, 4, 2615, 75, 224, 38, 524, 3911, 77, 38, 21, 4997, 7]
Decoded           : Top notch doctor in a top notch practice. Can't say I am surprised when I was referred to


In [6]:
# 4. Compute token-length statistics across the dataset before chunking
# Helpful to decide an appropriate block size
lengths = [len(ex["input_ids"]) for ex in tokenized_ds]
avg_len = sum(lengths) / len(lengths)
print(f"Avg tokens per review : {avg_len:.1f}")
print(f"Min / Max             : {min(lengths)} / {max(lengths)}")
print(f"Reviews under 512 tok : {sum(l <= 512 for l in lengths) / len(lengths):.1%}")

Avg tokens per review : 175.7
Min / Max             : 1 / 2348
Reviews under 512 tok : 95.6%


In [7]:
# 5. Concatenate all token sequences then slice into fixed-length blocks of 512 tokens
# Concatenating first avoids wasting capacity on short sequences
BLOCK_SIZE = 512

def chunk_into_blocks(examples):
    all_ids   = sum(examples["input_ids"], [])
    all_masks = sum(examples["attention_mask"], [])

    # Drop the remainder so every block is exactly BLOCK_SIZE
    keep = (len(all_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_ids, all_masks = all_ids[:keep], all_masks[:keep]

    chunked_ids   = [all_ids[i : i + BLOCK_SIZE]   for i in range(0, keep, BLOCK_SIZE)]
    chunked_masks = [all_masks[i : i + BLOCK_SIZE] for i in range(0, keep, BLOCK_SIZE)]

    return {"input_ids": chunked_ids, "attention_mask": chunked_masks}

lm_ds = tokenized_ds.map(chunk_into_blocks, batched=True, batch_size=500)
print(f"Total {BLOCK_SIZE}-token chunks : {len(lm_ds)}")

Map:   0%|          | 0/650000 [00:00<?, ? examples/s]

Total 512-token chunks : 222366


In [8]:
# 6. Wrap in a PyTorch DataLoader — batch size 32, shuffled
def collate_fn(samples):
    ids   = torch.tensor([s["input_ids"]      for s in samples], dtype=torch.long)
    masks = torch.tensor([s["attention_mask"] for s in samples], dtype=torch.long)
    # For causal LM the model takes labels = input_ids and shifts internally
    return {"input_ids": ids, "attention_mask": masks, "labels": ids.clone()}

train_loader = DataLoader(
    lm_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)
print(f"Total batches : {len(train_loader)}")

Total batches : 6949


In [9]:
# 7. Pull one batch and verify shapes, then decode a sequence back to text
for batch in train_loader:
    ids, masks, labels = batch["input_ids"], batch["attention_mask"], batch["labels"]
    print(f"input_ids shape    : {ids.shape}")
    print(f"attention_mask     : {masks.shape}")
    print(f"labels shape       : {labels.shape}")
    print(f"dtype              : {ids.dtype}")

    # Verify labels match input_ids (should be True)
    print(f"labels == input_ids: {torch.equal(ids, labels)}")

    # Decode the third sequence in the batch for a human-readable check
    print(f"\nDecoded sample (seq 2, first 60 tokens):\n{tokenizer.decode(ids[2][:60])}")
    break

input_ids shape    : torch.Size([32, 512])
attention_mask     : torch.Size([32, 512])
labels shape       : torch.Size([32, 512])
dtype              : torch.int64
labels == input_ids: True

Decoded sample (seq 2, first 60 tokens):
 in order to find this place (did anyone else have trouble too?) For a raw & organic cafe, I can't believe how tucked away it is from everything else on the west side of town! Sure, the lake view side is beautiful, but I'm a little puzzled at how this place gets
